<a href="https://colab.research.google.com/github/WhoisMonesh/Colab-Debrid-Downloader/blob/main/Colab-Debrid-Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-Debrid Multi-Source Downloader

Download torrents, direct links, and hoster files via Real-Debrid directly to Google Drive.


### 1. Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install Dependencies

In [0]:
!pip install requests -q
print('Dependencies installed.')

### 3. Configuration

In [0]:
# === CONFIGURATION ===
SAVE_PATH = '/content/downloads/RealDebridDownloader/'  # local temp dir
DRIVE_PATH = '/content/drive/My Drive/RealDebridDownloader/'  # final destination

API_TOKEN = ""  # Real-Debrid API token (https://real-debrid.com/apitoken)
MAGNET_LINK = ""  # Magnet link (torrent)
DIRECT_URLS = ""  # Comma-separated direct/hoster URLs (rapidgator, uploaded, etc.)

KEEP_ALIVE = True  # prevent Colab timeout

# === END CONFIGURATION ===

### 4. Download

In [0]:
import os, time, shutil
from google.colab import files

def format_bytes(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'

def get_all_files(root):
    result = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            result.append(os.path.join(dirpath, f))
    return result

def unrestrict_and_download(headers, link, save_path, progress_display, idx=0):
    r = requests.post('https://api.real-debrid.com/rest/1.0/unrestrict/link', headers=headers, data={'link': link})
    rj = r.json()
    dl_url = rj.get('download')
    fname = rj.get('filename', f'file_{idx}')
    sanitized = ''.join(c for c in fname if c not in '<>:"/\\|?*')
    fpath = os.path.join(save_path, sanitized)
    progress_display.update(HTML(f'<pre>Downloading ({idx+1}): {sanitized}</pre>'))
    r = requests.get(dl_url, stream=True)
    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    with open(fpath, 'wb') as f:
        for chunk in r.iter_content(8192):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = int(downloaded * 100 / total)
                    bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                    progress_display.update(HTML(f'<pre>Downloading ({idx+1}): |{bar}| {pct}% | {sanitized}</pre>'))
    return 1

def process_magnet(headers, magnet, save_path, progress_display):
    r = requests.post('https://api.real-debrid.com/rest/1.0/torrents/addMagnet', headers=headers, data={'magnet': magnet})
    rj = r.json()
    if 'id' not in rj:
        print(f'ERROR adding magnet: {rj}')
        return 0
    tid = rj['id']
    print(f'Torrent added: {tid}')
    requests.post(f'https://api.real-debrid.com/rest/1.0/torrents/selectFiles/{tid}', headers=headers, data={'files': 'all'})
    print('Waiting for Real-Debrid to process...')
    progress_display.update(HTML('<pre>Waiting for Real-Debrid to process torrent...</pre>'))
    while True:
        r = requests.get(f'https://api.real-debrid.com/rest/1.0/torrents/info/{tid}', headers=headers)
        status = r.json().get('status')
        if status == 'downloaded':
            break
        time.sleep(10)
    links = r.json().get('links', [])
    count = 0
    for link in links:
        count += unrestrict_and_download(headers, link, save_path, progress_display, count)
    return count

def main():
    from IPython.display import display, HTML, Javascript

    if not API_TOKEN:
        print('ERROR: Set API_TOKEN from https://real-debrid.com/apitoken')
        return
    if not MAGNET_LINK and not DIRECT_URLS:
        print('ERROR: Set MAGNET_LINK or DIRECT_URLS')
        return

    if KEEP_ALIVE:
        display(Javascript('''
            function keepAlive(){
                var btn=document.querySelector("colab-connect-button");
                if(btn)btn.click()
            }
            setInterval(keepAlive,120000);
        '''))
        print('Keep-alive active')

    os.makedirs(SAVE_PATH, exist_ok=True)
    print(f'Save path: {SAVE_PATH} (local)')

    import requests
    headers = {'Authorization': f'Bearer {API_TOKEN}'}

    begin = time.time()
    progress_display = display(HTML('<pre>Starting...</pre>'), display_id='dl-progress')
    total_files = 0

    if MAGNET_LINK:
        print(f'\nProcessing magnet: {MAGNET_LINK[:60]}...')
        total_files += process_magnet(headers, MAGNET_LINK, SAVE_PATH, progress_display)

    if DIRECT_URLS:
        urls = [u.strip() for u in DIRECT_URLS.split(',') if u.strip()]
        print(f'\nProcessing {len(urls)} direct URLs...')
        for i, url in enumerate(urls):
            print(f'  Unrestricting: {url[:80]}')
            total_files += unrestrict_and_download(headers, url, SAVE_PATH, progress_display, i)

    end = time.time()
    elapsed = int(end - begin)
    print()
    print('=' * 50)
    print('COMPLETE')
    print(f'Files: {total_files}')
    print(f'Elapsed: {elapsed // 60}m {elapsed % 60}s')
    print(f'Saved locally: {SAVE_PATH}')

    items = get_all_files(SAVE_PATH)
    total_sz = sum(os.path.getsize(f) for f in items)
    print(f'Downloaded {len(items)} files ({format_bytes(total_sz)})')

    if len(items) > 1:
        import zipfile
        processed = 0
        zpath = SAVE_PATH.rstrip('/').rstrip('\\') + '.zip'
        print(f'\nZipping {len(items)} files...')
        with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fp in items:
                arcname = os.path.relpath(fp, SAVE_PATH)
                zf.write(fp, arcname)
                processed += os.path.getsize(fp)
                pct = int(processed * 100 / total_sz) if total_sz else 0
                bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                progress_display.update(HTML(f'<pre>Zipping: |{bar}| {pct}% | {arcname}</pre>'))
        zip_name = os.path.basename(zpath)
        zip_size = os.path.getsize(zpath)
        print(f'Zip: {zip_name} ({format_bytes(zip_size)})')

    print(f'\nMoving to Drive...')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for f in os.listdir(SAVE_PATH):
        shutil.move(os.path.join(SAVE_PATH, f), os.path.join(DRIVE_PATH, f))
    print(f'Final: {DRIVE_PATH}')

    if len(items) > 1:
        drive_zip = os.path.join(DRIVE_PATH, zip_name)
        if os.path.exists(drive_zip):
            display(HTML(f'<p>Zip saved to Drive:<br><a href="{drive_zip}" download>{zip_name} ({format_bytes(zip_size)})</a></p>'))
            if zip_size < 500 * 1024 * 1024:
                files.download(drive_zip)
            else:
                print(f'Large file ({format_bytes(zip_size)}) — open Drive to download.')

    print('=' * 50)

if __name__ == '__main__':
    main()
